# 남은 9개 자료 수집 · 공개 사본

기록된 실행에서 후보 영상의 관절을 검사했으며, 9개 전부 독립 원본 기준에 미달해 새 모델 저장은 0개였습니다. [Colab 실행 확인](../docs/COLAB_RUNS.md)

이 사본은 당시 수집과 학습 코드를 보존하고 계정별 출력을 비웠습니다. 실행 시 본인의 Drive를 연결하며 `Exercise_AI_Models/Remaining9_New`에 요약 CSV를 씁니다. 이전 CSV와 구분하려면 코드의 `NEW` 경로에 새 실행 폴더를 지정하세요. 직접 촬영하거나 이용 허가를 받은 자료의 위치는 `INPUT`에서 지정합니다. 이 셀의 웹 후보와 검수된 정상·오류 자료는 별도로 관리합니다.


# 운동기구 26종: 미학습 9종 추가 데이터 확보 및 학습

**사용법:** `런타임 → 모두 실행` 한 번만 누르면 됩니다. Python 3.13 / MediaPipe Tasks 호환.

**자동 처리:** 기존 GitHub 운동 영상·GIF 데이터베이스와 스미스 스쿼트 직접 MP4 URL을 확인하고 다운로드 → 운동명 정확 일치 확인 → 관절 추출 → 독립 영상 8개 이상일 때 GOOD 기준 이상탐지 모델 학습 → Drive에 모델 및 CSV 보고서 저장. 기존 모델 덮어쓰기 금지.

**새로 조사한 헬스장 기구 영상 사이트:** Gymies Fitness, Tice Creek Fitness Center, Middlesex Health, Holland Township, ACE Fitness, 105Fitness, FitSite 등은 `manual_review_sources.csv`에 기록합니다. 사이트에 영상이 있다는 이유만으로 무단 다운로드하거나 GOOD 정답으로 확정하지 않습니다.

**추가 영상 자동 학습:** 직접 촬영했거나 학습 이용 허락을 받은 영상은 Drive `Exercise_AI_Training_Input/<exercise>/good/` 폴더에 넣으세요. 운동명은 코드의 `TARGETS` 키(예: `seated_leg_press`)를 사용합니다. 별도의 키 없이 자동으로 읽습니다. 승인된 직접 MP4 주소는 `Exercise_AI_Training_Input/approved_media_urls.csv`에 `exercise,url,label,source` 열로 넣을 수 있습니다.

**주의:** GOOD 시범은 정답 검증된 자료가 아닐 수 있습니다. BAD를 실제로 학습한 이진 분류 모델이 아니라 정상 패턴과의 차이를 측정하는 모델입니다. 실제 BAD 영상은 `bad` 폴더에 따로 보관하고 GOOD으로 섞지 마세요. 데이터가 부족하면 미학습으로 남으며, 모델을 만들었다고 정확도를 검증한 것은 아닙니다.

In [ ]:
# ONE CELL: setup + discover + download + pose + train + save + report
import os,sys,subprocess,json,re,hashlib,time,random,csv,shutil
from pathlib import Path
from urllib.parse import urlparse
subprocess.run([sys.executable,'-m','pip','-q','install','mediapipe>=0.10.30','opencv-python-headless','scikit-learn','joblib','requests','pandas','pillow'],check=True)
import requests,cv2,numpy as np,pandas as pd,joblib
from sklearn.ensemble import IsolationForest
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Exercise_AI_Models/Equipment26')
NEW=Path('/content/drive/MyDrive/Exercise_AI_Models/Remaining9_New')
NEW.mkdir(parents=True,exist_ok=True)
TEMP=Path('/content/exercise26_remaining9_temp'); TEMP.mkdir(parents=True,exist_ok=True)
# Optional: put personally recorded or licensed exercise videos here.
INPUT=Path('/content/drive/MyDrive/Exercise_AI_Training_Input')
INPUT.mkdir(parents=True,exist_ok=True)
# Input structure: Exercise_AI_Training_Input/<exercise>/good/*.mp4
# BAD clips are NOT relabeled GOOD. They remain reserved for supervised learning.

TARGETS={
 'treadmill_walk':['treadmill walk','treadmill walking','walking treadmill'],
 'recumbent_cycling':['recumbent bike','recumbent cycling','recumbent bicycle'],
 'elliptical_training':['elliptical trainer','elliptical machine','elliptical training'],
 'seated_chest_press':['seated chest press','machine seated chest press'],
 'machine_biceps_curl':['machine biceps curl','preacher curl machine','machine arm curl'],
 'rotary_torso':['rotary torso','torso rotation machine','seated torso rotation'],
 'seated_leg_press':['seated leg press','horizontal leg press','machine seated leg press'],
 'smith_squat':['smith machine squat','smith squat'],
 'double_twist':['double twist machine','double twist exercise machine']}
SOURCES={
 'exercise_video_db':'https://raw.githubusercontent.com/harshvishu/free-exercise-db-with-videos/main/data/exercises.json',
 'exercise_gifs':'https://raw.githubusercontent.com/hasaneyldrm/exercises-dataset/main/data/exercises.json',
 'workout_free':'https://raw.githubusercontent.com/namnq/workout-free-videos/main/data/exercises.json'}
SESSION=requests.Session(); SESSION.headers.update({'User-Agent':'Exercise26-Research/1.0'})
def fetch_json(url):
 try:
  r=SESSION.get(url,timeout=25);r.raise_for_status();return r.json()
 except Exception as e: print('메타데이터 접근 실패:',url,str(e)[:120]);return None
def entries(x):
 if isinstance(x,list):return x
 if isinstance(x,dict):
  for k in ('data','exercises','items','results'):
   if isinstance(x.get(k),list):return x[k]
 return []
def normalize(s):return re.sub(r'[^a-z0-9]+',' ',str(s).lower()).strip()
def name_match(name,aliases):
 fields=[normalize(name)]+[normalize(a) for a in aliases if isinstance(a,str)]
 for target,terms in TARGETS.items():
  for f in fields:
   if any(f==normalize(t) for t in terms):return target
 return None
def get_urls(obj):
 out=[]
 def walk(x,key=''):
  if isinstance(x,dict):
   for k,v in x.items():walk(v,k.lower())
  elif isinstance(x,list):
   for v in x:walk(v,key)
  elif isinstance(x,str) and x.startswith('http'):
   ext=urlparse(x).path.lower()
   if ext.endswith(('.mp4','.webm','.mov','.gif')) or any(k in key for k in ('video','gif','animation')):out.append(x)
 walk(obj)
 return list(dict.fromkeys(out))
def download(url,dst,max_mb=65):
 try:
  with SESSION.get(url,stream=True,timeout=35,allow_redirects=True) as r:
   r.raise_for_status();cl=int(r.headers.get('Content-Length','0'))
   if cl>max_mb*1024**2:return False
   size=0
   with open(dst,'wb') as f:
    for chunk in r.iter_content(65536):
     size+=len(chunk)
     if size>max_mb*1024**2:raise ValueError('too large')
     f.write(chunk)
  return size>5000
 except Exception as e:
  try:dst.unlink(missing_ok=True)
  except:pass
  return False
records=[]
# Additional discovered exercise-equipment demo sites are listed for manual review
# below; do not scrape video from their pages without checking permissions.
# Optional curated URLs CSV in Drive: exercise,url,label,source
manifest=INPUT/'approved_media_urls.csv'
if manifest.exists():
 for row in csv.DictReader(manifest.open(encoding='utf-8-sig',newline='')):
  exercise=(row.get('exercise') or '').strip()
  url=(row.get('url') or '').strip()
  label=(row.get('label') or '').strip().upper()
  if exercise in TARGETS and url.startswith('https://') and label in ('GOOD_ASSUMED_UNVERIFIED','GOOD_VERIFIED'):
   records.append(dict(exercise=exercise,source=row.get('source','user_approved'),name=exercise,url=url,label=label,status='candidate'))
# Verified direct MP4 from Future Fitness exercise library; exact Smith Machine Squat.
records.append(dict(
 exercise='smith_squat', source='futurefitness_direct_mp4',
 name='SMITH MACHINE SQUAT',
 url='https://futurefitness.co.uk/wp-content/uploads/sites/8/2025/06/Smith-Machine-Squat.mp4',
 label='GOOD_ASSUMED_UNVERIFIED', status='candidate'))
for source,url in SOURCES.items():
 data=fetch_json(url)
 es=entries(data)
 print(source,'메타데이터 항목:',len(es))
 for e in es:
  if not isinstance(e,dict):continue
  name=e.get('name') or e.get('title') or ''
  aliases=e.get('aliases',[])
  if not isinstance(aliases,list):aliases=[]
  target=name_match(name,aliases)
  if not target:continue
  urls=get_urls(e)
  for media in urls[:4]:records.append(dict(exercise=target,source=source,name=name,url=media,label='GOOD_ASSUMED_UNVERIFIED',status='candidate'))
# Explicitly do not use unrelated exercise examples as substitutes.
print('운동명 정확 일치 후보:',len(records))
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
MODEL_PATH=Path('/content/pose_landmarker_lite.task')
MODEL_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'
if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 100000:
 r=SESSION.get(MODEL_URL,timeout=120);r.raise_for_status();MODEL_PATH.write_bytes(r.content)
print('MediaPipe Tasks model:', MODEL_PATH,MODEL_PATH.stat().st_size,'bytes')
FEATURE_DIM=33*3

def make_detector(mode):
 options=vision.PoseLandmarkerOptions(
  base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
  running_mode=mode,num_poses=1,
  min_pose_detection_confidence=0.5,min_pose_presence_confidence=0.5,
  min_tracking_confidence=0.5)
 return vision.PoseLandmarker.create_from_options(options)

def pose_features(result):
 if not result.pose_landmarks:return None
 landmarks=result.pose_landmarks[0]
 if len(landmarks)!=33:return None
 a=np.asarray([[p.x,p.y,getattr(p,'visibility',0.0) or 0.0] for p in landmarks],dtype=np.float32)
 if (a[:,2]>.4).sum()<18:return None
 xy=a[:,:2].copy();center=(xy[23]+xy[24])/2
 scale=max(float(np.linalg.norm(xy[11]-xy[12])),float(np.linalg.norm(xy[23]-xy[24])),.05)
 a[:,:2]=(xy-center)/scale
 return a.flatten()

def to_mp_image(rgb):
 return mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(rgb))
def extract_video(path,max_frames=140):
 cap=cv2.VideoCapture(str(path))
 nframes=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
 fps=float(cap.get(cv2.CAP_PROP_FPS) or 0)
 if not np.isfinite(fps) or fps<=0:fps=30.0
 step=max(1,nframes//max_frames) if nframes else 5
 X=[];idx=0;last_ts=-1
 try:
  with make_detector(vision.RunningMode.VIDEO) as detector:
   while len(X)<max_frames:
    ok,frame=cap.read()
    if not ok:break
    current=idx;idx+=1
    if current%step:continue
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    ts=max(last_ts+1,int(current*1000/fps));last_ts=ts
    result=detector.detect_for_video(to_mp_image(rgb),ts)
    feature=pose_features(result)
    if feature is not None:X.append(feature)
 finally:cap.release()
 return np.asarray(X,dtype=np.float32).reshape(-1,FEATURE_DIM)

def extract_gif(path,max_frames=140):
 from PIL import Image,ImageSequence
 X=[]
 with Image.open(path) as im:
  total=getattr(im,'n_frames',1);step=max(1,total//max_frames)
  with make_detector(vision.RunningMode.IMAGE) as detector:
   for i,fr in enumerate(ImageSequence.Iterator(im)):
    if i%step:continue
    rgb=np.asarray(fr.convert('RGB'))
    result=detector.detect(to_mp_image(rgb))
    feature=pose_features(result)
    if feature is not None:X.append(feature)
 return np.asarray(X,dtype=np.float32).reshape(-1,FEATURE_DIM)
# Video counts as one independent recording regardless of number of frames.
collected={k:[] for k in TARGETS};seen_hash=set();download_count=0
for i,r in enumerate(records):
 if len(collected[r['exercise']])>=20:continue
 suffix=Path(urlparse(r['url']).path).suffix.lower()
 if suffix not in ('.mp4','.webm','.mov','.gif'):suffix='.mp4'
 path=TEMP/(f'{i:04d}'+suffix)
 if not download(r['url'],path):r['status']='download_failed';continue
 digest=hashlib.sha256(path.read_bytes()).hexdigest()
 if digest in seen_hash:r['status']='duplicate';path.unlink(missing_ok=True);continue
 seen_hash.add(digest);download_count+=1
 try:X=extract_gif(path) if suffix=='.gif' else extract_video(path)
 except Exception as e:
  print('관절 추출 실패:',r['exercise'],str(e)[:150]);X=np.empty((0,FEATURE_DIM),np.float32)
 r['pose_frames']=len(X)
 if len(X)<12:r['status']='insufficient_pose';continue
 r['status']='usable_good_assumed';collected[r['exercise']].append((r,X))
# Process user's own / licensed recordings. No API key required.
for target in TARGETS:
 folder=INPUT/target/'good'
 if not folder.exists(): continue
 for path in sorted(folder.iterdir()):
  if path.suffix.lower() not in ('.mp4','.webm','.mov','.gif'):continue
  if len(collected[target])>=40:break
  digest=hashlib.sha256(path.read_bytes()).hexdigest()
  if digest in seen_hash:continue
  seen_hash.add(digest)
  try:X=extract_gif(path) if path.suffix.lower()=='.gif' else extract_video(path)
  except Exception as e:
   print('Drive 입력 관절 추출 실패:',path.name,str(e)[:100]);continue
  if len(X)>=12:
   collected[target].append((dict(exercise=target,source='user_drive_good',url=str(path),status='usable_good_user_supplied',pose_frames=len(X)),X))
print('다운로드 성공:',download_count)
print('사용자 Drive 입력을 포함한 운동별 독립 원본 수:',{k:len(v) for k,v in collected.items()})
summary=[]
for target,items in collected.items():
 existing=list(ROOT.rglob(f'{target}*.joblib'))+list(ROOT.rglob(f'{target}*.pkl'))+list(NEW.glob(f'{target}*.joblib'))
 status='NOT_TRAINED';path=''
 if existing:status='EXISTING_MODEL_PRESERVED';path=str(existing[0])
 elif len(items)<8:status='INSUFFICIENT_INDEPENDENT_VIDEOS'
 else:
  # Cap contribution of each recording to avoid frame-count domination.
  rng=np.random.default_rng(42)
  X=np.vstack([a[rng.choice(len(a),size=min(90,len(a)),replace=False)] for _,a in items])
  if len(X)<120:status='INSUFFICIENT_POSE_FRAMES'
  else:
   model=IsolationForest(n_estimators=250,contamination=.07,random_state=42,n_jobs=-1)
   model.fit(X)
   obj={'model':model,'exercise':target,'type':'GOOD_ONLY_ANOMALY','features':'mediapipe_tasks_33_xy_visibility_normalized','independent_recordings':len(items),'pose_frames':len(X),'good_label':'ASSUMED_NOT_VERIFIED','bad_labeled_examples':0,'sources':[r['url'] for r,_ in items]}
   dest=NEW/(target+'_GOOD_ONLY.joblib');joblib.dump(obj,dest);status='TRAINED_GOOD_ONLY';path=str(dest)
 summary.append({'exercise':target,'independent_recordings':len(items),'pose_frames':sum(len(a) for _,a in items),'status':status,'model_path':path})
report=NEW/'data_collection_report.csv';pd.DataFrame(records).to_csv(report,index=False,encoding='utf-8-sig')
summary_path=NEW/'training_summary.csv';pd.DataFrame(summary).to_csv(summary_path,index=False,encoding='utf-8-sig')
print(pd.DataFrame(summary).to_string(index=False))
print('CSV 보고서:',report,summary_path)
print('기존 모델 삭제·덮어쓰기 없음. BAD 정답 학습 없음. 정확도 검증 미실시.')
# Manual review queue: websites confirmed to discuss the exact exercise, but not
# scraped as training videos without verifying access, rights and exact equipment.
review_rows=[
 ('seated_leg_press','https://www.acefitness.org/resources/everyone/exercise-library/154/seated-leg-press/','instruction/reference only'),
 ('seated_leg_press','https://www.105fitness.online/how-to-videos','video page; manual review'),
 ('seated_leg_press','https://www.gymiesfitness.com/equipment','gym equipment demo; manual review'),
 ('seated_leg_press','https://www.ticefitnesscenter.com/equipment-videos/','gym equipment demo; manual review'),
 ('seated_chest_press','https://www.gymiesfitness.com/equipment','gym equipment demo; manual review'),
 ('seated_chest_press','https://www.ticefitnesscenter.com/equipment-videos/','gym equipment demo; manual review'),
 ('seated_chest_press','https://middlesexhealth.org/learning-center/life-stages-and-living-well/weight-machine-exercises','video page; manual review'),
 ('machine_biceps_curl','https://middlesexhealth.org/learning-center/life-stages-and-living-well/weight-machine-exercises','video page; manual review'),
 ('rotary_torso','https://smartworkout.app/en/exercise-library/abs/rotary-torso','instruction/reference only'),
 ('rotary_torso','https://www.ticefitnesscenter.com/equipment-videos/','gym equipment demo; manual review'),
 ('rotary_torso','https://www.youtube.com/watch?v=myQy1DhRBUQ','video page; manual rights review'),
 ('double_twist','https://www.fitsite.com/demos.html','machine side twists reference; equipment verification required'),
 ('double_twist','https://www.windomhealth.org/','hip twister candidate; exact equipment verification required'),
 ('treadmill_walk','https://github.com/RepDB/exercise-dataset','equipment metadata; not real video'),
 ('treadmill_walk','https://www.gymiesfitness.com/equipment','gym equipment demo; manual review'),
 ('treadmill_walk','https://www.puregym.com/','gym equipment demo; manual review'),
 ('recumbent_cycling','https://github.com/RepDB/exercise-dataset','equipment metadata; not real video'),
 ('recumbent_cycling','https://www.gymiesfitness.com/equipment','gym equipment demo; manual review'),
 ('elliptical_training','https://github.com/RepDB/exercise-dataset','equipment metadata; not real video'),
 ('elliptical_training','https://www.ticefitnesscenter.com/equipment-videos/','gym equipment demo; manual review'),
 ('smith_squat','https://www.gymiesfitness.com/equipment','gym equipment demo; manual review')]
pd.DataFrame(review_rows,columns=['exercise','source_url','reason']).to_csv(NEW/'manual_review_sources.csv',index=False,encoding='utf-8-sig')
print('검토 필요 자료:',NEW/'manual_review_sources.csv')
# Source media remains in /content; clean after training to save space.
shutil.rmtree(TEMP,ignore_errors=True)

# Temporary source files removed; trained models and CSV reports remain in Drive.
shutil.rmtree(TEMP,ignore_errors=True)
